### Chain Using LangGraph
Build a simple chain using Langgraph that uses 4 important concepts

- How to use chat messages as our graph state
- How to use chat models in graph nodes
- How to bind tools to our LLM in chat models
- How to execute the tools call in our graph nodes 

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
os.environ["GEMINI_API_KEY"]=os.getenv("GEMINI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


#### How to use chat messages as our graph state
##### Messages

We can use messages which can be used to capture different roles within a conversation.
LangChain has various message types including HumanMessage, AIMessage, SystemMessage and ToolMessage.
These represent a message from the user, from chat model, for the chat model to instruct behavior, and from a tool call.

Every message have these important components.

- content - content of the message
- name - Specify the name of author
- response_metadata - optionally, a dict of metadata (e.g., often populated by model provider for AIMessages)



In [2]:
from langchain_core.messages import AIMessage,HumanMessage
from pprint import pprint

messages=[AIMessage(content=f"Please tell me how can I help",name="LLMModel")]
messages.append(HumanMessage(content=f"I want to learn coding",name="Yuval"))
messages.append(AIMessage(content=f"Which programming language you want to learn",name="LLMModel"))
messages.append(HumanMessage(content=f"I want to learn python programming language",name="Yuval"))

for message in messages:
    message.pretty_print()
    

================================== Ai Message ==================================
Name: LLMModel

Please tell me how can I help
================================ Human Message =================================
Name: Yuval

I want to learn coding
================================== Ai Message ==================================
Name: LLMModel

Which programming language you want to learn
================================ Human Message =================================
Name: Yuval

I want to learn python programming language


### Chat Models

We can use the sequence of message as input with the chatmodels using LLM's.

In [3]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="openai/gpt-oss-20b")
result=llm.invoke(messages)
print(result)

content='### 🎉 Welcome to the world of Python!  \nBelow is a **road‑map** you can follow at your own pace, plus a list of the best resources, tools, and practice ideas to help you master Python from scratch to real‑world projects.\n\n---\n\n## 1️⃣ What You’ll Get Out of This Guide\n\n| Goal | What You’ll Learn | Why It Matters |\n|------|------------------|----------------|\n| **Syntax & fundamentals** | Variables, data types, loops, functions, modules | The building blocks of every program |\n| **Problem‑solving** | Conditional logic, list comprehensions, recursion | Turn a problem into code |\n| **Libraries & tools** | `pip`, virtual environments, Jupyter notebooks | Work efficiently & share your work |\n| **Project‑based learning** | Build a CLI, a web scraper, a simple web app | Apply skills to something useful |\n| **Best practices** | PEP\u202f8, docstrings, unit tests | Write clean, maintainable code |\n\n---\n\n## 2️⃣ Your 6‑Week Learning Plan\n\n> **Tip:** Aim for **1–2 hours 

In [4]:
result.response_metadata

{'token_usage': {'completion_tokens': 1719,
  'prompt_tokens': 113,
  'total_tokens': 1832,
  'completion_time': 1.997127766,
  'completion_tokens_details': {'reasoning_tokens': 30},
  'prompt_time': 0.005436517,
  'prompt_tokens_details': None,
  'queue_time': 0.050151763,
  'total_time': 2.002564283},
 'model_name': 'openai/gpt-oss-20b',
 'system_fingerprint': 'fp_c5a89987dc',
 'service_tier': 'on_demand',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'groq'}

### Tools
Tools can be integrated with the LLM models to interact with external systems. External systems can be API's, third party tools.

Whenever a query is asked the model can choose to call the tool and this query is based on the 
natural language input and this will return an output that matches the tool's schema

In [5]:
def add(a:int,b:int)-> int:
    """ Add a and b
    Args:
        a (int): first int
        b (int): second int

    Returns:
        int
    """
    return a+b

In [6]:
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A033CF56A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A033CF6120>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [7]:
### Binding tool with llm

llm_with_tools = llm.bind_tools([add])

tool_call = llm_with_tools.invoke([HumanMessage(content=f"What is 2 plus 2",name="Yuval")])

In [9]:
tool_call.tool_calls

[{'name': 'add',
  'args': {'a': 2, 'b': 2},
  'id': 'fc_7883e66a-5c7b-4311-bbfe-6965c0d0f27f',
  'type': 'tool_call'}]

### Using messages as state

In [10]:
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage

class State(TypedDict):
    message:list[AnyMessage]

#### Reducers
Now, we have a minor problem!

As we discussed, each node will return a new value for our state key messages.

But, this new value will override the prior messages value.

As our graph runs, we want to append messages to our messages state key.

We can use reducer functions to address this.

Reducers allow us to specify how state updates are performed.

If no reducer function is specified, then it is assumed that updates to the key should override it as we saw before.

But, to append messages, we can use the pre-built add_messages reducer.

This ensures that any messages are appended to the existing list of messages.

We simply need to annotate our messages key with the add_messages reducer function as metadata.

In [11]:
from langgraph.graph.message import add_messages
from typing import Annotated
class State(TypedDict):
    messages:Annotated[list[AnyMessage],add_messages]

### Reducers with add_messages

In [15]:
initial_messages=[AIMessage(content=f"Please tell me how can I help",name="LLMModel")]
initial_messages.append(HumanMessage(content=f"I want to learn coding",name="Yuval"))
initial_messages

[AIMessage(content='Please tell me how can I help', additional_kwargs={}, response_metadata={}, name='LLMModel', tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I want to learn coding', additional_kwargs={}, response_metadata={}, name='Yuval')]

In [16]:
ai_message=AIMessage(content=f"Which programming language you want to learn",name="LLMModel")
ai_message

AIMessage(content='Which programming language you want to learn', additional_kwargs={}, response_metadata={}, name='LLMModel', tool_calls=[], invalid_tool_calls=[])

In [17]:
### Reducers add_messages is to append instead of override
add_messages(initial_messages,ai_message)

[AIMessage(content='Please tell me how can I help', additional_kwargs={}, response_metadata={}, name='LLMModel', id='eee5024a-1379-498b-ae7c-165be31afaef', tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I want to learn coding', additional_kwargs={}, response_metadata={}, name='Yuval', id='a254614d-bb76-4f14-b509-3f6b7f4b40e5'),
 AIMessage(content='Which programming language you want to learn', additional_kwargs={}, response_metadata={}, name='LLMModel', id='9972015b-2ddf-446d-b586-dd4b87f78973', tool_calls=[], invalid_tool_calls=[])]